# Choosing the smoothing parameter

This notebook compares **classical in-sample CV, GCV, AICc, and BIC** as implemented by the library. These scores choose a smoother; they do **not** evaluate future forecasting performance.

The reference mean $\mu$ is fixed at zero in all examples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import TrendSmoother, select_lambda, plot_selection, plot_trend

rng = np.random.default_rng(7)
t = np.linspace(0, 1, 70)
y = 3 * t + 0.5 * np.sin(4 * np.pi * t) + rng.normal(0, 0.3, len(t))
solver = TrendSmoother(len(y), order=2)

## Compare four established criteria

All selectors evaluate the same series and differencing order. Finite penalty candidates are searched on a geometric grid; the infinite-penalty endpoint is also considered.

In [ ]:
criteria = ["cv", "gcv", "aicc", "bic"]
results = {name: select_lambda(solver, y, criterion=name, mu=0.0) for name in criteria}
for name, result in results.items():
    print(f"{name.upper():>4}: lambda={result.lambda_:>12.4g}, smoothness={result.smoothness:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
for ax, name in zip(axes.flat, criteria):
    plot_selection(results[name], ax=ax)
    ax.set_title(f"{name.upper()} selector")
plt.show()

## Inspect the selected GCV trend

The [SelectionResult API](../api.html) exposes the candidate penalties, scores, optimal penalty, and normalized smoothness. Different criteria need not select the same smoothing level.

In [ ]:
gcv = results["gcv"]
fit = solver.fit(y, lambda_=gcv.lambda_, mu=0.0)
plot_trend(y, fit, title="Trend selected by GCV")
plt.show()

## Important statistical boundary

For `mu="data"`, the reference mean is itself estimated from the observations. The package intentionally does **not** offer the classical selectors in that mode as exact leave-one-out calculations. Their direct formulas require **fixed numeric** $\mu$. The displayed criterion scores are in-sample smoothing diagnostics, not future prediction scores.